In [1]:
import pandas as pd
from pathlib import Path

orders = pd.read_csv('../data/raw/olist_orders_dataset.csv')
reviews = pd.read_csv('../data/raw/olist_order_reviews_dataset.csv')


# Colunas de orders 
- order_id - mantém object
- order_status - mantém object
- order_delivered_customer_date > datetime
- order_purchase_timestamp > datetime

# Colunas de reviews
- review_id - mantém object
- order_id (chave) - mantém object
- review_score - mantém int
- review_answer_timestamp > datetime


In [2]:
#investigando as colunas de orders

orders_cln = orders.copy()
orders_cln['dt_entrega'] = pd.to_datetime(orders_cln['order_delivered_customer_date'], errors='coerce')
print('A quantidade de nulos da coluna order_delivered_customer_date é: ')
print(orders_cln['dt_entrega'].isna().sum())

orders_cln['dt_compra'] = pd.to_datetime(orders_cln['order_purchase_timestamp'], errors='coerce')
print('A quantidade de nulos da coluna das datas de compra é:')
print(orders_cln['dt_compra'].isna().sum())

A quantidade de nulos da coluna order_delivered_customer_date é: 
2965
A quantidade de nulos da coluna das datas de compra é:
0


In [3]:
entregues = orders_cln[orders_cln['order_status'] == 'delivered'].copy()

entregues_cln = entregues[entregues['dt_entrega'].notna()].copy()

nulos_entregue = entregues['dt_entrega'].isna().sum()
limpos_entregue = len(entregues_cln)

assert len(entregues) - limpos_entregue == 8
print(f"Total de nulos da coluna dt_entregue: {nulos_entregue}")
print(f"Total de linhas da tabela limpa: {limpos_entregue}")
print(f"Total de linhas da tabela não limpa {len(entregues)}")
print(f"O assert deve ser igual, e o total é: {nulos_entregue + limpos_entregue}")

Total de nulos da coluna dt_entregue: 8
Total de linhas da tabela limpa: 96470
Total de linhas da tabela não limpa 96478
O assert deve ser igual, e o total é: 96478


In [4]:
#Investigando as colunas de reviews
reviews_cln = reviews.copy()
reviews_cln['data_resposta'] = pd.to_datetime(reviews_cln['review_answer_timestamp'], errors='coerce')
print('A quantidade de nulos na coluna de data de reviews é de:')
print(reviews_cln['data_resposta'].isna().sum())

A quantidade de nulos na coluna de data de reviews é de:
0


In [5]:
#contabilizando os duplicados
uniq = reviews_cln['order_id'].nunique()
print(uniq)
print(len(reviews_cln))
print(f"A diferença é de {len(reviews_cln) - uniq}")


98673
99224
A diferença é de 551


In [6]:
#removendo os duplicados
reviews_ord = reviews_cln.sort_values('data_resposta', ascending=True)
reviews_dedup = reviews_ord.drop_duplicates(subset='order_id', keep='last')

Aqui removemos os 551 reviews de order_id duplicados pelo seguinte critério:
consideramos que a data do review mais recente era a mais importante pois ela representa
a opinião final do cliente a respeito da plataforma, logo, é o melhor indicador a respeito 
da satisfação atual do cliente.

Uma ressalva: a última review está mais distante da entrega no tempo, então embute algum ruído pós-entrega

In [7]:
assert len(reviews_dedup) == uniq